# Neural Network SMS Text Classifier 📱

Classifies messages as ham or spam, returning both the spam probability and a label. Uses a TensorFlow text-vectorization, embedding and bidirectional LSTM pipeline.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import numpy as np
import pandas as pd
import tensorflow as tf

tf.keras.utils.set_random_seed(42)
urls = {
    "train-data.tsv": "https://cdn.freecodecamp.org/project-data/sms/train-data.tsv",
    "valid-data.tsv": "https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv",
}
for filename,url in urls.items():
    if not Path(filename).exists():
        urlretrieve(url,filename)
train = pd.read_csv("train-data.tsv",sep="\t",header=None,names=["label","message"])
valid = pd.read_csv("valid-data.tsv",sep="\t",header=None,names=["label","message"])
print("Training:",train.shape,"Validation:",valid.shape)
print(train.label.value_counts())


## Prepare the labels and text
Spam maps to 1 and normal ham maps to 0. A vocabulary is learned only from the training split.

In [ ]:
X_train = np.asarray(train.message.fillna("").astype(str).tolist(),dtype=str)
y_train = (train.label.astype(str) == "spam").astype("float32").to_numpy()
X_valid = np.asarray(valid.message.fillna("").astype(str).tolist(),dtype=str)
y_valid = (valid.label.astype(str) == "spam").astype("float32").to_numpy()
vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=12000, output_mode="int", output_sequence_length=90)
vectorizer.adapt(X_train)


## Train the neural network
A learned embedding and bidirectional LSTM model word order in SMS messages.

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(),dtype=tf.string),
    vectorizer,
    tf.keras.layers.Embedding(12000,64,mask_zero=True),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(32)),
    tf.keras.layers.Dense(32,activation="relu"),
    tf.keras.layers.Dropout(.35),
    tf.keras.layers.Dense(1,activation="sigmoid")
])
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=.001),
              loss="binary_crossentropy",metrics=["accuracy"])
history = model.fit(X_train,y_train,validation_data=(X_valid,y_valid),
    batch_size=64,epochs=12,verbose=1,
    callbacks=[tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",patience=3,restore_best_weights=True)])
print("Validation loss and accuracy:",model.evaluate(X_valid,y_valid,verbose=0))


## Predict a single message
The API follows freeCodeCamp: `[spam_probability, 'ham' or 'spam']`. The threshold is 0.5.

In [ ]:
def predict_message(message):
    probability = float(model.predict(np.array([str(message)]),verbose=0)[0][0])
    return [probability, "spam" if probability >= .5 else "ham"]

print(predict_message("Hey, are you available for coffee this afternoon?"))
print(predict_message("You won a free prize! Claim your reward now!"))


## Official-style test messages
Run after training. A pass is shown only if all seven held-out challenge examples are classified correctly.

In [ ]:
test_messages = [
    "how are you doing today",
    "sale today! to stop texts call 98912460324",
    "i dont want to go. can we try it a different day? available sat",
    "our new mobile video service is live. just install on your phone to start watching.",
    "you have won £1000 cash! call to claim your prize.",
    "i'll bring it tomorrow. don't forget the milk.",
    "wow, is your arm alright. that happened to me one time too",
]
expected = ["ham","spam","ham","spam","spam","ham","ham"]
predictions = [predict_message(text) for text in test_messages]
for text, result in zip(test_messages,predictions):
    print(result, " | ", text[:65])
passed = all(prediction[1] == label
             for prediction,label in zip(predictions,expected))
print("PASS" if passed else "NOT YET PASSING — tune and retrain")
